In [2]:
 ##IMPORTS & PATH SETUP
import yfinance as yf
import pandas as pd
import os
import sys
import time

# Go up one level from notebooks/ to reach the project root where config.py lives
sys.path.insert(0, os.path.abspath('..'))
import config

print(f"yfinance version : {yf.__version__}")
print(f"Stocks to fetch  : {len(config.NIFTY_50)} NIFTY + {len(config.SP500_TOP30)} S&P = {len(config.NIFTY_50) + len(config.SP500_TOP30)} total")
print(f"Date range       : {config.START_DATE} → {config.END_DATE}")

yfinance version : 1.4.1
Stocks to fetch  : 50 NIFTY + 30 S&P = 80 total
Date range       : 2010-01-01 → 2024-12-31


In [3]:
##DOWNLOAD DATA
def download_stock(ticker, save_dir):
    """
    Downloads historical OHLCV data for a single stock via yfinance
    and saves it as a CSV file.

    auto_adjust=True  → Close column becomes adjusted close (accounts
                         for dividends and stock splits). This gives us
                         true return calculations later.
    """
    try:
        df = yf.download(
            ticker,
            start   = config.START_DATE,
            end     = config.END_DATE,
            auto_adjust = True,
            progress    = False    # suppress yfinance's own progress bar
        )

        if df.empty:
            print(f"  [SKIP]  {ticker:<22} — no data returned")
            return False

        # yfinance >= 0.2.x sometimes returns MultiIndex columns even for
        # a single ticker. Flatten it to a simple column list.
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] for col in df.columns]

        # Sanitise ticker for a safe filename
        # M&M.NS → MANDM_NS.csv  |  BAJAJ-AUTO.NS → BAJAJ_AUTO_NS.csv
        safe_name = (ticker
                     .replace("&", "AND")
                     .replace(".", "_")
                     .replace("-", "_"))
        filepath = os.path.join(save_dir, f"{safe_name}.csv")

        df.to_csv(filepath)

        rows      = len(df)
        date_from = df.index[0].date()
        date_to   = df.index[-1].date()
        print(f"  [OK]    {ticker:<22} | {rows:>4} rows | {date_from} → {date_to}")
        return True

    except Exception as e:
        print(f"  [FAIL]  {ticker:<22} | {e}")
        return False

In [4]:
## NIFTY 50
print("=" * 60)
print("  NIFTY 50 — Indian market (NSE)")
print("=" * 60)

india_dir   = "../data/raw/india"
success_in  = []
failed_in   = []

for ticker in config.NIFTY_50:
    ok = download_stock(ticker, india_dir)
    (success_in if ok else failed_in).append(ticker)
    time.sleep(0.5)   # small pause — avoids Yahoo rate-limiting

print(f"\n  Downloaded : {len(success_in)} / {len(config.NIFTY_50)}")
if failed_in:
    print(f"  Failed     : {failed_in}")

  NIFTY 50 — Indian market (NSE)
  [OK]    TCS.NS                 | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    INFY.NS                | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    HCLTECH.NS             | 3700 rows | 2010-01-04 → 2024-12-30
  [OK]    WIPRO.NS               | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    TECHM.NS               | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    HDFCBANK.NS            | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    ICICIBANK.NS           | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    AXISBANK.NS            | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    KOTAKBANK.NS           | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    SBIN.NS                | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    INDUSINDBK.NS          | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    BAJFINANCE.NS          | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    BAJAJFINSV.NS          | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    HDFCLIFE.NS            | 1756 rows

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


  [SKIP]  TATAMOTORS.NS          — no data returned
  [OK]    M&M.NS                 | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    BAJAJ-AUTO.NS          | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    HEROMOTOCO.NS          | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    EICHERMOT.NS           | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    SUNPHARMA.NS           | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    DRREDDY.NS             | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    CIPLA.NS               | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    DIVISLAB.NS            | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    APOLLOHOSP.NS          | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    RELIANCE.NS            | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    ONGC.NS                | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    BPCL.NS                | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    POWERGRID.NS           | 3699 rows | 2010-01-04 → 2024-12-30
  [OK]    NTPC.NS        

In [5]:
##S&P 500 TOP 30
print("=" * 60)
print("  S&P 500 Top 30 — US market (NYSE / NASDAQ)")
print("=" * 60)

us_dir     = "../data/raw/us"
success_us = []
failed_us  = []

for ticker in config.SP500_TOP30:
    ok = download_stock(ticker, us_dir)
    (success_us if ok else failed_us).append(ticker)
    time.sleep(0.3)

print(f"\n  Downloaded : {len(success_us)} / {len(config.SP500_TOP30)}")
if failed_us:
    print(f"  Failed     : {failed_us}")

  S&P 500 Top 30 — US market (NYSE / NASDAQ)
  [OK]    AAPL                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    NVDA                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    MSFT                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    AVGO                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    AMD                    | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    ORCL                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    CSCO                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    CRM                    | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    AMZN                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    TSLA                   | 3651 rows | 2010-06-29 → 2024-12-30
  [OK]    HD                     | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    MCD                    | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    COST                   | 3773 rows | 2010-01-04 → 2024-12-30
  [OK]    GOOGL                 

In [6]:
##SUMMARY
print("=" * 60)
print("  VERIFICATION SUMMARY")
print("=" * 60)

india_files = [f for f in os.listdir(india_dir) if f.endswith('.csv')]
us_files    = [f for f in os.listdir(us_dir)    if f.endswith('.csv')]

print(f"\n  Files in data/raw/india/ : {len(india_files)}")
print(f"  Files in data/raw/us/    : {len(us_files)}")
print(f"  Total CSV files          : {len(india_files) + len(us_files)}")

# ── Row count range ───────────────────────────────────────────
def row_stats(folder, files):
    counts = [len(pd.read_csv(os.path.join(folder, f))) for f in files]
    return min(counts), max(counts), sum(counts) // len(counts), sum(counts)

mn, mx, avg, total_in = row_stats(india_dir, india_files)
print(f"\n  India row counts  — min: {mn}, max: {mx}, avg: {avg}, total: {total_in:,}")

mn, mx, avg, total_us = row_stats(us_dir, us_files)
print(f"  US row counts     — min: {mn}, max: {mx}, avg: {avg}, total: {total_us:,}")

print(f"\n  Grand total rows  : {total_in + total_us:,}")

# ── Spot check — one from each market ────────────────────────
print("\n  Sample — TCS.NS (first 3 rows):")
print(pd.read_csv(f"{india_dir}/TCS_NS.csv", index_col=0, parse_dates=True).head(3).to_string())

print("\n  Sample — AAPL (first 3 rows):")
print(pd.read_csv(f"{us_dir}/AAPL.csv", index_col=0, parse_dates=True).head(3).to_string())

  VERIFICATION SUMMARY

  Files in data/raw/india/ : 49
  Files in data/raw/us/    : 30
  Total CSV files          : 79

  India row counts  — min: 1756, max: 3700, avg: 3616, total: 177,187
  US row counts     — min: 3174, max: 3773, avg: 3748, total: 112,469

  Grand total rows  : 289,656

  Sample — TCS.NS (first 3 rows):
                 Close        High         Low        Open   Volume
Date                                                               
2010-01-04  259.259155  261.759826  258.362351  260.345641  1963682
2010-01-05  259.328033  261.983921  257.499962  260.414540  2014488
2010-01-06  253.464432  259.448806  252.826338  259.328071  3349176

  Sample — AAPL (first 3 rows):
               Close      High       Low      Open     Volume
Date                                                         
2010-01-04  6.406481  6.421150  6.357687  6.389119  493729600
2010-01-05  6.417558  6.453780  6.383731  6.424144  601904800
2010-01-06  6.315479  6.443004  6.308893  6.417559  